# Réplica de Sensei: chatbots en producción (RQ2)

**Artículo replicado:** de Lara, J., del Pozzo, A., Guerra, E., & Sánchez Cuadrado, J. (2026). *Automated end-to-end testing for conversational agents*. Journal of Systems and Software, 233, 112685. https://doi.org/10.1016/j.jss.2025.112685

**Qué se replica:** la RQ2 del paper (*"Is Sensei effective to find defects in conversational agents in the wild?"*) con la **misma herramienta** (código original de Sensei), los **mismos perfiles**, las **mismas reglas** y chatbots de la Tabla 6:

| Chatbot | Cómo se replica | Conversaciones |
|---|---|---|
| Julie (Amtrak) | En vivo: Sensei conversa con el chatbot real | 32 |
| Serviceform | En vivo | 12 |
| Catalina | El chatbot ya no permite el acceso (error 403). Se reproduce el análisis sobre las conversaciones originales de los autores | 14 |
| SAIC | Igual que Catalina (403) | 14 |

**Qué se compara con el paper:**
- Fallas por tipo (Tablas 6 y 7): C = caídas, G = objetivos no cumplidos, T = timeouts, L = bucles, GR1 = fuera de tema, GR2 = respuestas repetidas, GR3 = no responde en el idioma del usuario, GR4 = enlaces rotos o botones vacíos.
- Tiempo de respuesta del chatbot (promedio, mínimo y máximo), que Sensei registra en cada turno.

**Diferencia principal con el paper:** el paper usó gpt-4o y gpt-4o-mini (OpenAI, de pago). Aquí se usa **Gemini 3.5 Flash-Lite**, gratuito, a través de su API compatible con OpenAI.

**Requisito:** una clave gratuita de Gemini (https://aistudio.google.com/apikey) guardada en Colab: icono de llave 🔑 (*Secrets*), nombre `TESIS`, con *Notebook access* activado.

**Cuota gratuita:** unas 500 llamadas al día; se reinicia a las 3:00 a.m. (hora de Lima). Una ejecución completa no entra en un solo día:
- **Día 1:** Pasos 1 a 6 y Serviceform en los Pasos 7 y 8. Descarga el zip (Paso 10).
- **Día 2:** Pasos 1 a 5, restaura el zip (Paso 11), Julie en los Pasos 7 y 8, y luego los Pasos 9 y 10.
- **Demo:** Pasos 1 a 5 y Paso 12 (unas 20 llamadas).

Ejecuta las celdas en orden, de arriba hacia abajo.

## Paso 1. Descargar Sensei y el paquete de replicación de los autores

Se fija la versión exacta (commit) para que la réplica sea reproducible.

**Qué hacer:** dale ▶. Tarda menos de 1 minuto.

**Qué deberías ver:** los archivos de Sensei (`src`, `examples`…) y la lista de chatbots de los autores (`catalina`, `julie`, `saic_malaga`, `serviceform`…).

In [ ]:
%cd /content
!rm -rf sensei use
!git clone -q https://github.com/sensei-chat/sensei.git sensei
!git -C sensei checkout -q eba960fcdca056c4a1ef736b9a276a4b7625d8c3
!git clone -q https://github.com/satori-chatbots/user-simulator-evaluation.git use
!git -C use checkout -q 28a0d6ac1f0330b4e49d813164cec609bc7a9748
!ls sensei use/chatbots-wild-analysis/chatbots

## Paso 2. Instalar las dependencias de Sensei

Se instala en un entorno aparte (Python 3.12) con las versiones de `requirements.txt` de Sensei, para no chocar con las librerías de Colab.

**Qué hacer:** dale ▶ y espera (1 a 5 minutos).

**Qué deberías ver:** `OK  openai 1.52.2 | httpx 0.27.0`.

**Si falla:** vuelve a ejecutar la celda. Si Colab ofrece "reiniciar sesión", **no** reinicies.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 /content/venv
# requirements.txt de Sensei viene en UTF-16: se convierte a UTF-8
texto = open("/content/sensei/requirements.txt", "rb").read().decode("utf-16")
open("/content/requirements_sensei.txt", "w", encoding="utf-8").write(texto)
!uv pip install -q --python /content/venv/bin/python -r /content/requirements_sensei.txt
PY = "/content/venv/bin/python"
!{PY} -c "import langchain_openai, openai, httpx; print('OK  openai', openai.__version__, '| httpx', httpx.__version__)"

## Paso 3. Configurar la API gratuita (Gemini)

Sensei usa la librería de OpenAI. Gemini ofrece una API compatible con OpenAI, así que basta con cambiar la dirección (`OPENAI_BASE_URL`) y la clave.

**Qué hacer:** ejecuta las 3 celdas en orden. Si Colab pregunta por el acceso al secreto, acepta.

**Qué deberías ver:** `Configurado. Modelo: gemini-3.5-flash-lite`, la lista de modelos "lite" y las 3 pruebas (`hola`, `POSITIVE`, `{'answer': 'París'}`) con `Todo OK`.

**Si falla:**
- `404` o `model not found`: elige un nombre de la lista (sin "preview"), ponlo en `MODELO` y repite las 3 celdas.
- `401` o `API key not valid`: el secreto `TESIS` no existe o no tiene *Notebook access*.
- `429 ... per day`: se acabó la cuota del día; espera a las 3:00 a.m.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    CLAVE = userdata.get("TESIS")
except Exception:
    CLAVE = getpass.getpass("Pega tu clave de Gemini: ")

MODELO = "gemini-3.5-flash-lite"   # gratuito, ~500 llamadas/día
URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

os.environ["OPENAI_API_KEY"] = CLAVE
os.environ["OPENAI_BASE_URL"] = URL
os.environ["OPENAI_API_BASE"] = URL
# Carpeta con el parche de reintentos (Paso 4) + código de Sensei
os.environ["PYTHONPATH"] = "/content/parches:/content/sensei/src"
print("Configurado. Modelo:", MODELO)

In [ ]:
%%writefile /content/prueba_api.py
# Prueba que el modelo soporta las 3 formas en que Sensei llama a la API.
import sys, json
from openai import OpenAI
modelo = sys.argv[1]
c = OpenAI()
print("Modelos 'lite' disponibles:", [m.id for m in c.models.list() if "lite" in m.id][:15])
r = c.chat.completions.create(model=modelo, messages=[{"role": "user", "content": "Responde solo: hola"}])
print("1) chat normal:", r.choices[0].message.content.strip())
s = c.chat.completions.create(model=modelo, messages=[{"role": "user", "content": "Responde solo: POSITIVE"}], stream=True)
print("2) streaming:", "".join(ch.choices[0].delta.content or "" for ch in s if ch.choices).strip())
esquema = {"type": "json_schema", "json_schema": {"name": "x", "strict": True, "schema": {
    "type": "object", "required": ["answer"], "additionalProperties": False,
    "properties": {"answer": {"type": ["string", "null"], "description": "la capital de Francia"}}}}}
j = c.chat.completions.create(model=modelo, messages=[{"role": "user", "content": "¿Cuál es la capital de Francia?"}],
                              response_format=esquema)
print("3) JSON estructurado:", json.loads(j.choices[0].message.content))
print("\nTodo OK: el modelo sirve para Sensei.")

In [ ]:
!{PY} /content/prueba_api.py {MODELO}

## Paso 4. Ajustes mínimos (documentados)

No se cambia la lógica de evaluación de Sensei. Ajustes:

1. **Modelo:** donde el código o los perfiles dicen `gpt-4o`, `gpt-4o-mini`, `gpt-4o-2024-08-06` u `o1-mini`, se pone el modelo gratuito.
2. **Reintentos:** si el plan gratuito responde "demasiadas solicitudes" (429), se reintenta hasta 8 veces en lugar de 2.
3. **El usuario habla primero** (`is_starter: False`): los conectores de Julie y Serviceform no implementan el saludo inicial del chatbot (`execute_starter_chatbot`), así que con `True` Sensei se detiene con `NotImplementedError`.
4. **Corrección de un bug de Sensei** (`sensei-chat.py`): cuando el usuario habla primero, el código original envía su primer mensaje **dos veces** al chatbot. Se agrega un `continue` para enviarlo una sola vez. Los autores no lo notaron porque en sus perfiles hablaba primero el chatbot.
5. **Perfiles:** nombre de prueba único por perfil (si no, las conversaciones se sobrescriben), el formato de lista antiguo de un perfil de Julie se pasa al actual, `ask_about` se renombra a `goals` en 2 perfiles de SAIC y se corrige el `test_name` de `catalina_padron.yml`.

**Qué hacer:** dale ▶.

**Qué deberías ver:** `Llamadas al modelo cambiadas en el código: 6`, `Bug del primer mensaje corregido: True` y los perfiles de los 4 chatbots (Julie 4, Serviceform 4, Catalina 3, SAIC 4).

**Si falla:** ejecuta antes el Paso 3 (ahí se define `MODELO`).

In [ ]:
import re, pathlib
os.makedirs("/content/parches", exist_ok=True)
open("/content/parches/sitecustomize.py", "w").write('''
try:
    import openai
    _init_original = openai.OpenAI.__init__
    def _init(self, *args, **kwargs):
        kwargs["max_retries"] = max(kwargs.get("max_retries") or 0, 8)
        _init_original(self, *args, **kwargs)
    openai.OpenAI.__init__ = _init
except Exception:
    pass
''')

# 1) Modelo en el código de Sensei
patron = re.compile(r'"(gpt-4o-2024-08-06|gpt-4o-mini|gpt-4o|o1-mini)"')
cambios = 0
for f in pathlib.Path("/content/sensei/src").rglob("*.py"):
    t = f.read_text(encoding="utf-8")
    n = patron.sub(f'"{MODELO}"', t)
    if n != t:
        cambios += len(patron.findall(t))
        f.write_text(n, encoding="utf-8")
print("Llamadas al modelo cambiadas en el código:", cambios)

# 4) Bug: el primer mensaje del usuario se enviaba dos veces
p = pathlib.Path("/content/sensei/src/sensei-chat.py")
t = p.read_text(encoding="utf-8")
a = "                    print_chatbot(response)\n                    bot_starter = True\n"
if a + "                    continue\n" not in t:
    t = t.replace(a, a + "                    continue\n", 1)
    p.write_text(t, encoding="utf-8")
print("Bug del primer mensaje corregido:", a + "                    continue\n" in t)

# 3) y 5) Perfiles de los autores
for chatbot in ["julie", "serviceform", "catalina", "saic_malaga"]:
    origen = pathlib.Path(f"/content/sensei/examples/profiles/{chatbot}")
    destino = pathlib.Path(f"/content/perfiles/{chatbot}")
    destino.mkdir(parents=True, exist_ok=True)
    for f in sorted(origen.glob("*.yml")):
        t = f.read_text(encoding="utf-8")
        t = re.sub(r"(\n\s*model:\s*)\S+", rf"\g<1>{MODELO}", t)
        t = t.replace("is_starter: True", "is_starter: False")
        t = t.replace("\n  ask_about:", "\n  goals:")
        t = re.sub(r"\n  - (number|goal_style|interaction_style):", r"\n  \1:", t)
        if chatbot in ["julie", "serviceform"]:
            t = re.sub(r'test_name:\s*"[^"]*"', f'test_name: "{f.stem}"', t)
        if f.name == "catalina_padron.yml":
            t = t.replace('test_name: "saic_malaga_padron"', 'test_name: "catalina_padron"')
        (destino / f.name).write_text(t, encoding="utf-8")
    print(chatbot, "->", sorted(x.name for x in destino.glob("*.yml")))

## Paso 5. Probar la conexión con los chatbots

Se envía un mensaje a cada chatbot con el mismo conector que usa Sensei (caja negra: solo pregunta y respuesta). No gasta cuota de Gemini.

**Qué deberías ver:** `Julie: OK` y `Serviceform: OK` con un pedazo de la respuesta. Catalina y SAIC muestran `HTTP 403`: el proveedor (1MillionBot) ya no acepta la clave de acceso que trae Sensei. Es un hallazgo de la réplica, no un error del notebook.

**Si falla:** si Julie o Serviceform no responden, repite una vez. Si siguen sin responder, ese chatbot cambió desde el paper: anótalo como resultado.

In [ ]:
%%writefile /content/prueba_chatbots.py
import requests
from technologies.chatbot_connectors import JulieChatbot, ChatbotServiceform, ChatbotCatalinaRivas, ChatbotSaicMalaga

for nombre, clase, msg in [("Julie", JulieChatbot, "Hi, I want to travel from Boston to New York"),
                           ("Serviceform", ChatbotServiceform, "Hello, what plans do you offer?")]:
    try:
        ok, resp = clase("x").execute_with_input(msg)
        print(f"{nombre}: {'OK' if ok else 'ERROR'} ->", (resp or "")[:250].replace("\n", " "), "\n")
    except Exception as e:
        print(f"{nombre}: NO RESPONDE ->", repr(e)[:250], "\n")

# Catalina y SAIC (1MillionBot): se muestra el código HTTP que devuelve el servidor
for nombre, clase in [("Catalina", ChatbotCatalinaRivas), ("SAIC", ChatbotSaicMalaga)]:
    try:
        bot = clase("x")
        r1 = requests.post(bot.reset_url, headers=bot.headers, json=bot.reset_payload, timeout=30)
        r2 = requests.post(bot.url, headers=bot.headers, json=bot.payload, timeout=30)
        print(f"{nombre}: reinicio HTTP {r1.status_code} | mensaje HTTP {r2.status_code} ->", r2.text[:150].replace("\n", " "), "\n")
    except Exception as e:
        print(f"{nombre}: NO RESPONDE ->", repr(e)[:250], "\n")

In [ ]:
!cd /content/sensei && {PY} /content/prueba_chatbots.py

## Paso 6. Catalina y SAIC: reproducir el análisis con las conversaciones de los autores

Como Catalina y SAIC dan 403, se aplican **las reglas de los autores** a **sus conversaciones originales** (`use/chatbots-wild-analysis/output/`). Si el resultado coincide con el CSV de los autores, Sensei-check está bien instalado y reproduce la Tabla 6.

**Qué hacer:** dale ▶ (menos de 1 minuto, unas 30 llamadas).

**Qué deberías ver:** para cada chatbot, la tabla de los autores y la de verificación. Deben coincidir, salvo quizás `only_talks_about`, que la decide el LLM (Gemini en lugar de gpt-4o).

In [ ]:
for chatbot in ["catalina", "saic_malaga"]:
    get_ipython().system(
        f"cd /content/sensei && {PY} src/sensei-check.py "
        f"--rules /content/use/chatbots-wild-analysis/chatbots/{chatbot}/rules "
        f"--conversations /content/use/chatbots-wild-analysis/output/{chatbot} --dump /content/verificacion_{chatbot}.csv")
    print(f"\n===== {chatbot} =====")
    print("Autores:\n", open(f"/content/use/chatbots-wild-analysis/output/{chatbot}/{chatbot}_test_results.csv").read())
    print("Verificación:\n", open(f"/content/verificacion_{chatbot}.csv").read())

## Paso 7. Generar las conversaciones en vivo (Sensei-chat)

Igual que los autores: cada chatbot se prueba con 2 personalidades, **normal** y **directa** (`personalities/direct-user.yml`). Salen 32 conversaciones con Julie y 12 con Serviceform, como en la Tabla 6.

**Qué hacer:** en `CHATBOTS` deja solo el chatbot del día (por la cuota) y dale ▶.
- Serviceform: unos 8 minutos y 150 llamadas.
- Julie: 20 a 30 minutos y unas 400 llamadas.

**Qué deberías ver:** la conversación en pantalla (`User: …` / `Chatbot: …`) y, al final de cada bloque, los tiempos de respuesta (`Average assistant response time…`).

**Si falla:** `429 ... per minute`: espera 1 minuto y repite. `429 ... per day`: se acabó la cuota del día.

In [ ]:
CHATBOTS = ["serviceform"]        # día 1: ["serviceform"]   día 2: ["julie"]
for chatbot in CHATBOTS:
    for personalidad in ["normal", "direct"]:
        extra = "" if personalidad == "normal" else "--personality personalities/direct-user.yml"
        print(f"\n========== {chatbot} | personalidad {personalidad} ==========")
        get_ipython().system(
            f"cd /content/sensei && {PY} src/sensei-chat.py --technology {chatbot} --chatbot x "
            f"--user /content/perfiles/{chatbot} --extract /content/salida/{chatbot}/{personalidad} {extra}")

## Paso 8. Revisar las conversaciones (Sensei-check)

Se aplican **las reglas de los autores** (`use/chatbots-wild-analysis/chatbots/<chatbot>/rules`).

**Qué hacer:** dale ▶ (usa la misma lista `CHATBOTS` del Paso 7).

**Qué deberías ver:** una tabla `rule, checks, pass, fail…`. `checks` debe ser 32 (Julie) o 12 (Serviceform).

In [ ]:
for chatbot in CHATBOTS:
    print(f"\n========== {chatbot} ==========")
    get_ipython().system(
        f"cd /content/sensei && {PY} src/sensei-check.py "
        f"--rules /content/use/chatbots-wild-analysis/chatbots/{chatbot}/rules "
        f"--conversations /content/salida/{chatbot} --dump /content/salida/{chatbot}/{chatbot}_test_results.csv")
    print(open(f"/content/salida/{chatbot}/{chatbot}_test_results.csv").read())

## Paso 9. Comparar con el paper

- **Paper:** CSV y conversaciones originales de los autores. Sus totales coinciden con la Tabla 6: Julie 53, Serviceform 9, Catalina 20 y SAIC 22.
- **Réplica:** lo obtenido en los Pasos 6 a 8.

La regla `chatbot_fills_all_slots` no aparece en las Tablas 6 y 7, por eso no se cuenta en el total.

**Qué hacer:** dale ▶ cuando tengas los resultados de Julie y Serviceform (si uno falta, se salta).

**Qué deberías ver:** una tabla de fallas por tipo por chatbot (columnas *Paper* y *Réplica*), los tiempos de respuesta y un resumen final. El resumen se guarda en `salida/resumen_replica.csv`.

In [ ]:
import glob, yaml, pandas as pd

MAPEO = {"no_response": "C", "goal_not_completed": "G", "timeout": "T", "exceeded_loop_limit": "L",
         "only_talks_about": "GR1", "chatbot_does_not_repeat": "GR2",
         "chatbot_responds_in_same_language": "GR3", "non_empty_links": "GR4"}
ORDEN = ["C", "G", "T", "L", "GR1", "GR2", "GR3", "GR4"]
AUTORES = "/content/use/chatbots-wild-analysis/output"

def fallas(csv):
    df = pd.read_csv(csv)
    df["codigo"] = df["rule"].map(MAPEO)
    df = df.dropna(subset=["codigo"])          # chatbot_fills_all_slots no está en la Tabla 6
    return df.set_index("codigo")["fail"].reindex(ORDEN), int(df["checks"].max())

def tiempos(carpeta):
    t = []
    for f in glob.glob(f"{carpeta}/**/*.yml", recursive=True):
        if "__report__" in f:
            continue
        for d in yaml.safe_load_all(open(f, encoding="utf-8")):
            if isinstance(d, dict) and "assistant response time" in d:
                t += [x for x in d["assistant response time"] if x is not None]
    s = pd.Series(t, dtype=float)
    return {"promedio (s)": round(s.mean(), 2), "mínimo (s)": round(s.min(), 2),
            "máximo (s)": round(s.max(), 2), "respuestas medidas": len(s)}

def comparar(titulo, csv_a, csv_b, col_b):
    fa, na = fallas(csv_a)
    fb, nb = fallas(csv_b)
    tabla = pd.DataFrame({"Paper": fa, col_b: fb})
    tabla.loc["TOTAL"] = tabla.sum(min_count=1)
    print(f"\n=== {titulo} | conversaciones: paper {na}, {col_b.lower()} {nb}")
    display(tabla.astype("Int64"))
    return fa, fb, nb

resumen = []
for chatbot, nombre in [("julie", "Julie (Amtrak)"), ("serviceform", "Serviceform")]:
    csv = f"/content/salida/{chatbot}/{chatbot}_test_results.csv"
    if not os.path.exists(csv):
        print(f"\n(Falta {nombre}: ejecuta los Pasos 7 y 8 con ese chatbot o restaura el zip en el Paso 11)")
        continue
    fa, fb, n = comparar(f"{nombre} - en vivo", f"{AUTORES}/{chatbot}/{chatbot}_test_results.csv", csv, "Réplica")
    t = pd.DataFrame({"Paper": tiempos(f"{AUTORES}/{chatbot}"), "Réplica": tiempos(f"/content/salida/{chatbot}")})
    print("Tiempo de respuesta del chatbot:")
    display(t)
    resumen.append({"Chatbot": nombre, "Tipo": "En vivo", "Conversaciones": n,
                    "Fallas paper (Tabla 6)": int(fa.sum()), "Fallas réplica": int(fb.sum()),
                    "Tipos con falla (réplica)": ", ".join(fb[fb > 0].index),
                    "Tiempo promedio paper (s)": t.loc["promedio (s)", "Paper"],
                    "Tiempo promedio réplica (s)": t.loc["promedio (s)", "Réplica"]})

for chatbot, nombre in [("catalina", "Catalina"), ("saic_malaga", "SAIC")]:
    ver = f"/content/verificacion_{chatbot}.csv"
    if not os.path.exists(ver):
        print(f"\n(Falta {ver}: ejecuta el Paso 6)")
        continue
    fa, fb, n = comparar(f"{nombre} - análisis reproducido (en vivo: 403)",
                         f"{AUTORES}/{chatbot}/{chatbot}_test_results.csv", ver, "Reproducción")
    resumen.append({"Chatbot": nombre, "Tipo": "Análisis reproducido (403 en vivo)", "Conversaciones": n,
                    "Fallas paper (Tabla 6)": int(fa.sum()), "Fallas réplica": int(fb.sum()),
                    "Tipos con falla (réplica)": ", ".join(fb[fb > 0].index),
                    "Tiempo promedio paper (s)": None, "Tiempo promedio réplica (s)": None})

resumen = pd.DataFrame(resumen)
display(resumen)
os.makedirs("/content/salida", exist_ok=True)
resumen.to_csv("/content/salida/resumen_replica.csv", index=False)

## Paso 10. Descargar los resultados

**Qué hacer:** dale ▶. Se descarga `resultados_replica_sensei.zip` (conversaciones, CSV, resumen y perfiles). Guárdalo: es la evidencia de la réplica y sirve para restaurar en otro día (Paso 11).

**Si no se descarga:** acepta el aviso de "descargas múltiples" del navegador, o descárgalo desde el panel de Archivos (icono de carpeta) con clic derecho.

In [ ]:
!cd /content && rm -f resultados_replica_sensei.zip && zip -qr resultados_replica_sensei.zip salida perfiles verificacion_*.csv
from google.colab import files
files.download("/content/resultados_replica_sensei.zip")

## Paso 11. Restaurar los resultados de otro día

Colab borra todo al desconectarse. Para recuperar lo de días anteriores:

**Qué hacer:**
1. Abre el panel de Archivos (icono de carpeta, a la izquierda) y arrastra ahí `resultados_replica_sensei.zip` (debe quedar en `/content`).
2. Dale ▶.

**Qué deberías ver:** la lista de chatbots restaurados (por ejemplo, `serviceform`).

**Si falla:** `cannot find` significa que el zip no está en `/content` o tiene otro nombre (por ejemplo, `resultados_replica_sensei (1).zip`): renómbralo.

In [ ]:
!cd /content && unzip -o -q resultados_replica_sensei.zip && ls /content/salida

## Paso 12. Demo en vivo (1 conversación real)

Muestra el método completo en pocos minutos: Sensei-chat conversa con el chatbot real y Sensei-check aplica las reglas de los autores a esa conversación. Usa unas 20 llamadas.

**Qué hacer:** ejecuta antes los Pasos 1 a 5. Luego dale ▶. Si Julie no responde ese día, cambia `DEMO_CHATBOT` a `"serviceform"`.

**Qué deberías ver:** la conversación (`User:` / `Chatbot:`), el tiempo de respuesta y la tabla de reglas con `checks = 1`.

In [ ]:
import shutil
DEMO_CHATBOT = "julie"     # o "serviceform"
PERFIL = {"julie": "julie_travel_schedule.yml", "serviceform": "change_language.yml"}[DEMO_CHATBOT]
DEMO = "/content/demo"
shutil.rmtree(DEMO, ignore_errors=True)
pathlib.Path(f"{DEMO}/perfil").mkdir(parents=True)
t = pathlib.Path(f"/content/perfiles/{DEMO_CHATBOT}/{PERFIL}").read_text(encoding="utf-8")
t = re.sub(r"(\n\s*number:\s*)\d+", r"\g<1>1", t)          # una sola conversación
(pathlib.Path(f"{DEMO}/perfil") / PERFIL).write_text(t, encoding="utf-8")
print(f"=== 1) Sensei-chat: el usuario simulado (LLM) conversa con {DEMO_CHATBOT} ===")
get_ipython().system(f"cd /content/sensei && {PY} src/sensei-chat.py --technology {DEMO_CHATBOT} --chatbot x "
                     f"--user {DEMO}/perfil --extract {DEMO}/salida")
print("\n=== 2) Sensei-check: reglas de los autores ===")
get_ipython().system(f"cd /content/sensei && {PY} src/sensei-check.py "
                     f"--rules /content/use/chatbots-wild-analysis/chatbots/{DEMO_CHATBOT}/rules "
                     f"--conversations {DEMO}/salida --dump {DEMO}/resultado.csv")
if os.path.exists(f"{DEMO}/resultado.csv"):
    print(open(f"{DEMO}/resultado.csv").read())
else:
    print("No se generó la conversación: el chatbot no respondió. Cambia DEMO_CHATBOT y vuelve a ejecutar.")

## Cómo leer los resultados

- **Fallas por tipo:** si la réplica encuentra los mismos tipos de falla que el paper, la herramienta se comporta igual. Los números exactos pueden variar porque el usuario simulado es un LLM (responde distinto cada vez) y porque los chatbots pueden haber cambiado desde 2024.
- **Tiempo de respuesta:** el paper no lo reporta por chatbot, pero sus conversaciones originales lo guardan. Por eso se compara con el de la réplica.
- **Diferencias declaradas:** Gemini 3.5 Flash-Lite en lugar de gpt-4o/gpt-4o-mini; fecha de ejecución distinta (2026); el usuario habla primero; corrección del bug del primer mensaje; Catalina y SAIC solo con el análisis reproducido (403 en vivo).